# update 方法使用示例

演示如何在分箱器训练完成后手工修改切分点。

参考 toad.Combiner.update 方法设计。

In [1]:
import os, sys
sys.path.append('../')

import numpy as np
import pandas as pd
from hscredit.core.binning import OptimalBinning
from hscredit.utils.datasets import germancredit

## 1. 加载数据

In [2]:
# 加载数据
df = germancredit().copy()
y = df['class'].astype(int)
X = df[['age_in_years', 'credit_amount', 'duration_in_month', 'purpose']].copy()

print(f'数据形状: {X.shape}')
print(f'特征列表: {X.columns.tolist()}')

数据形状: (1000, 4)
特征列表: ['age_in_years', 'credit_amount', 'duration_in_month', 'purpose']


## 2. 初始分箱（使用 user_splits）

In [3]:
# 使用 user_splits 初始化分箱器
user_splits = {
    'age_in_years': [25, 35, 45, 55],
    'credit_amount': [1000, 3000, 6000, 10000],
    'duration_in_month': [6, 12, 24, 36],
    'purpose': [['car'], ['furniture'], ['radio/tv'], ['domestic appliances']],
}
binner = OptimalBinning(user_splits=user_splits)
binner.fit(X, y)

print('初始切分点:')
for feature in X.columns:
    print(f'  {feature}: {binner.splits_[feature]}')

初始切分点:
  age_in_years: [25. 29. 35.]
  credit_amount: [ 3000.  4720.  6000. 10000.]
  duration_in_month: [12. 24. 36. 48.]
  purpose: [['car'], ['furniture'], ['radio/tv'], ['domestic appliances']]


## 3. 只更新切分点（不重新计算统计表）

In [4]:
# 只更新切分点（不重新计算统计表）
binner.update({
    'age_in_years': [20, 30, 40, 50, 60],  # 增加一个切分点
})
print('更新后的 age_in_years 切分点:', binner.splits_['age_in_years'])

更新后的 age_in_years 切分点: [20. 30. 40. 50. 60.]


## 4. 更新切分点并重新计算统计表

In [5]:
# 更新切分点并重新计算统计表
binner.update({
    'credit_amount': [2000, 5000, 8000],  # 修改切分点
    'purpose': [['car', 'furniture'], ['radio/tv', 'domestic appliances']],  # 合并类别
}, X=X, y=y)

print('更新后的切分点:')
for feature in ['credit_amount', 'purpose']:
    print(f'  {feature}: {binner.splits_[feature]}')

更新后的切分点:
  credit_amount: [2000. 5000. 8000.]
  purpose: [['car', 'furniture'], ['radio/tv', 'domestic appliances']]


## 5. 查看更新后的分箱统计表

In [6]:
# 查看更新后的分箱统计表
bin_table = binner.get_bin_table('credit_amount')
bin_table[['分箱', '分箱标签', '样本总数', '坏样本率']]

,分箱,分箱标签,样本总数,坏样本率
0,0,"[-inf, 2000)",432,0.280093
1,1,"[2000, 5000)",380,0.265789
2,2,"[5000, 8000)",118,0.338983
3,3,"[8000, +inf)",70,0.542857


## 6. 链式调用

In [7]:
# 链式调用
X_binned = binner.update({
    'duration_in_month': [12, 24, 36]
}).transform(X.head(5))

print('分箱后的数据:')
X_binned

分箱后的数据:


,age_in_years,credit_amount,duration_in_month,purpose
0,5,0,0,0
1,1,2,3,0
2,3,1,1,0
3,3,2,3,0
4,4,1,2,0


## 7. 将数值型特征改为类别型分箱

In [8]:
# 将数值型特征改为类别型分箱
# 注意：这会将特征从数值型改为类别型
binner.update({
    'age_in_years': [['20-30', '30-40'], ['40-50'], ['50-60']],  # 改为类别型分箱
})

print(f"更新后的 age_in_years 类型: {binner.feature_types_['age_in_years']}")
print(f"更新后的 age_in_years 分箱: {binner.splits_['age_in_years']}")

更新后的 age_in_years 类型: categorical
更新后的 age_in_years 分箱: [['20-30', '30-40'], ['40-50'], ['50-60']]


## 8. 完整 API 参考

```python
def update(
    self,
    splits_dict: Dict[str, Union[List, List[List]]],
    X: Optional[Union[pd.DataFrame, np.ndarray]] = None,
    y: Optional[Union[pd.Series, np.ndarray]] = None,
) -> 'BaseBinning'
```

**参数说明：**
- `splits_dict`: 新的切分点字典
  - 数值型: `{'age': [25, 35, 45, 55]}`
  - 类别型: `{'city': [['北京', '上海'], ['广州', '深圳']]`
- `X`: 可选，训练数据。如果提供，会重新计算分箱统计表
- `y`: 可选，目标变量

**返回：** self，支持链式调用